# Qualidade dos Dados e Tratamento

Este notebook realiza o processo de **Data Quality** e **Tratamento** sobre os dados simulados do projeto ***Turnover Crítico***.

Nesta etapa iremos:

- Validar a integridade dos dados (linhas, tipos, ranges)
- Detectar valores nulos, incoerentes ou extremos
- Garantir consistência temporal e relacional entre as tabelas
- Criar variáveis derivadas importantes para análises e hipóteses
- Preparar a base final para EDA e modelagem

O foco é equilibrar **racional técnico** com **racional de negócio**, garantindo que as bases reflitam cenários realistas de People Analytics no setor financeiro.

## Imports e carregamento dos dados

In [16]:
import pandas as pd
import numpy as np
from datetime import date

# Carregar datasets simulados a partir da pasta raw
colab = pd.read_csv("../data/raw/dim_colaboradores.csv")
rh_mensal = pd.read_csv("../data/raw/fato_rh_mensal.csv")
promocoes = pd.read_csv("../data/raw/eventos_promocoes.csv")
desligamentos = pd.read_csv("../data/raw/eventos_desligamentos.csv")

## Análise Exploratória dos Dados

### Validar carregamento

In [17]:
display(colab.head(1))
display(rh_mensal.head(1))
display(promocoes.head(1))
display(desligamentos.head(1))

,employee_id,area,nivel,genero,raca,data_admissao,idade,salario_base_inicial
0,1,Tecnologia,Sênior,F,Parda,2018-09-25,37,11366.628604


,employee_id,mes_ano,engajamento,performance,horas_extras
0,1,2024-01-01,66.406476,2.336944,12.857885


,employee_id,data,nivel_anterior,nivel_novo
0,1,2024-05-01,Sênior,Líder


,employee_id,data_desligamento,tipo,custo_reposicao
0,42,2024-02-01,Involuntário,35928.607687


### Schema das tabelas e tipo de dados

In [18]:
print("dim_colaboradores:", colab.shape)
print("fato_rh_mensal:", rh_mensal.shape)
print("eventos_promocoes:", promocoes.shape)
print("eventos_desligamentos:", desligamentos.shape)

colab.info()
rh_mensal.info()
promocoes.info()
desligamentos.info()

dim_colaboradores: (2500, 8)
fato_rh_mensal: (60000, 5)
eventos_promocoes: (350, 4)
eventos_desligamentos: (130, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   employee_id           2500 non-null   int64  
 1   area                  2500 non-null   object 
 2   nivel                 2500 non-null   object 
 3   genero                2500 non-null   object 
 4   raca                  2500 non-null   object 
 5   data_admissao         2500 non-null   object 
 6   idade                 2500 non-null   int64  
 7   salario_base_inicial  2500 non-null   float64
dtypes: float64(1), int64(2), object(5)
memory usage: 156.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   emplo

### Dados ausentes/nulos

In [19]:
print("Nulos por tabela:")
print("\nDim Colaboradores:")
print(colab.isna().sum())

print("\nFato RH Mensal:")
print(rh_mensal.isna().sum())

print("\nPromoções:")
print(promocoes.isna().sum())

print("\nDesligamentos:")
print(desligamentos.isna().sum())

Nulos por tabela:

Dim Colaboradores:
employee_id             0
area                    0
nivel                   0
genero                  0
raca                    0
data_admissao           0
idade                   0
salario_base_inicial    0
dtype: int64

Fato RH Mensal:
employee_id     0
mes_ano         0
engajamento     0
performance     0
horas_extras    0
dtype: int64

Promoções:
employee_id       0
data              0
nivel_anterior    0
nivel_novo        0
dtype: int64

Desligamentos:
employee_id          0
data_desligamento    0
tipo                 0
custo_reposicao      0
dtype: int64


### Tratamento de Dados (tipos e datas)

In [20]:
# Converte datas relevantes
rh_mensal["mes_ano"] = pd.to_datetime(rh_mensal["mes_ano"])
promocoes["data"] = pd.to_datetime(promocoes["data"])
desligamentos["data_desligamento"] = pd.to_datetime(desligamentos["data_desligamento"])
colab["data_admissao"] = pd.to_datetime(colab["data_admissao"])

### Checar a consistência de tempo dos dados

- Por exemplo, o desligamento não pode ser antes da demissão, se houver isso, os dados estão com algum tipo de erro/inconsistência

In [21]:
erros_adm = desligamentos[
    desligamentos["data_desligamento"] < colab.set_index("employee_id").loc[
        desligamentos["employee_id"], "data_admissao"
    ].values
]

print("Desligamentos com datas inválidas:", erros_adm.shape[0])

Desligamentos com datas inválidas: 9


A célula acima informa que temos 9 desligamentos com datas inválidas, nesse caso, pode ter sido algum erro de cadastro ou de digitação. Ás vezes, em cenários reais, ocorrem por erros de integração de sistemas, normalização, erros em termos de datas. Neste caso, como é um projeto fictício e a ocorrência tem baixa frequência, optarei pela exclusão desses 9 casos.

In [22]:
# --- TRATAMENTO DE DESLIGAMENTOS INVÁLIDOS ---

# 1) Garantir que as datas estão em datetime (idempotente, caso já tenha convertido)
colab["data_admissao"] = pd.to_datetime(colab["data_admissao"], errors="coerce")
desligamentos["data_desligamento"] = pd.to_datetime(desligamentos["data_desligamento"], errors="coerce")

# 2) Trazer a data de admissão para a tabela de desligamentos (merge por employee_id)
desli = desligamentos.merge(
    colab[["employee_id", "data_admissao"]],
    on="employee_id",
    how="left",
    validate="m:1"
)

# 3) Criar flag de invalidade temporal
desli["flag_invalido_temporal"] = desli["data_desligamento"] < desli["data_admissao"]

# 4) Quantificar casos inválidos (esperado: 9, conforme seu diagnóstico)
qtd_invalidos = desli["flag_invalido_temporal"].sum()
print(f"Desligamentos inválidos detectados: {qtd_invalidos}")

# 5) Inspecionar opcionalmente (head) os casos inválidos
if qtd_invalidos > 0:
    display(desli.loc[desli["flag_invalido_temporal"], 
                      ["employee_id", "data_admissao", "data_desligamento", "tipo"]].head())

# 6) Excluir registros inválidos
desli_corrigido = desli.loc[~desli["flag_invalido_temporal"]].copy()

# 7) Sanity check pós-tratamento: deve retornar 0
qtd_invalidos_pos = (desli_corrigido["data_desligamento"] < desli_corrigido["data_admissao"]).sum()
print(f"Inválidos após tratamento (esperado=0): {qtd_invalidos_pos}")

# 8) Atualizar objeto 'desligamentos' com a versão corrigida (sem colunas auxiliares)
desligamentos = desli_corrigido[
    ["employee_id", "data_desligamento", "tipo", "custo_reposicao"]
].reset_index(drop=True)

print(f"Formato final de 'desligamentos' após limpeza: {desligamentos.shape}")

Desligamentos inválidos detectados: 9


,employee_id,data_admissao,data_desligamento,tipo
3,131,2024-12-05,2024-10-01,Voluntário
8,223,2024-08-23,2024-01-01,Voluntário
49,1058,2024-05-13,2024-05-01,Voluntário
62,1287,2024-06-19,2024-01-01,Involuntário
63,1320,2024-06-18,2024-01-01,Voluntário


Inválidos após tratamento (esperado=0): 0
Formato final de 'desligamentos' após limpeza: (121, 4)


### Criação de variáveis - Feature Engineering

In [23]:
# 1) Tempo de casa
colab["data_admissao"] = pd.to_datetime(colab["data_admissao"], errors="coerce")
hoje = pd.Timestamp.today().normalize()
tempo_meses = ((hoje - colab["data_admissao"]).dt.days // 30)
colab["tempo_casa_meses"] = tempo_meses.fillna(0).clip(lower=0).astype(int)

In [24]:
# 2) Engajamento médio por colaborador
eng_medio = (
    rh_mensal.groupby("employee_id")["engajamento"]
    .mean()
    .rename("engajamento_medio")
)

In [25]:
# 3) Horas extras média
hex_medio = (
    rh_mensal.groupby("employee_id")["horas_extras"]
    .mean()
    .rename("horas_extras_media")
)

In [26]:
# 4) Performance média
perf_medio = (
    rh_mensal.groupby("employee_id")["performance"]
    .mean()
    .rename("performance_media")
)

In [27]:
# Agora vou juntar essas variáveis na tabela de colaboradores
colab = colab.merge(eng_medio, on="employee_id", how="left")
colab = colab.merge(hex_medio, on="employee_id", how="left")
colab = colab.merge(perf_medio, on="employee_id", how="left")
colab.head(1)

,employee_id,area,nivel,genero,raca,data_admissao,idade,salario_base_inicial,tempo_casa_meses,engajamento_medio,horas_extras_media,performance_media
0,1,Tecnologia,Sênior,F,Parda,2018-09-25,37,11366.628604,90,65.452687,19.11853,3.030411


### Verificação de outliers

In [28]:
# Função para verificar a existência

def detectar_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    return ((series < limite_inferior) | (series > limite_superior)).sum()

print("Outliers em horas extras:", detectar_outliers(colab["horas_extras_media"]))
print("Outliers em engajamento:", detectar_outliers(colab["engajamento_medio"]))
print("Outliers em performance:", detectar_outliers(colab["performance_media"]))

Outliers em horas extras: 363
Outliers em engajamento: 0
Outliers em performance: 7


In [29]:
# Tratamento, para truncar valores extremos para limites "aceitáveis", ou seja, nesse caso estou truncanto
# os 1% maiores e 1% menores valores
'''
Se o valor é < P1 (1º percentil) → vira P1
Se o valor é > P99 (99º percentil) → vira P99
O resto permanece igual
'''

def winsorizar(serie, p=0.01):
    minimo = serie.quantile(p)
    maximo = serie.quantile(1-p)
    return serie.clip(minimo, maximo)

colab["horas_extras_media"] = winsorizar(colab["horas_extras_media"])
colab["engajamento_medio"] = winsorizar(colab["engajamento_medio"])
colab["performance_media"] = winsorizar(colab["performance_media"])


print("Outliers após tratamento:")
print("Horas extras:", detectar_outliers(colab["horas_extras_media"]))
print("Engajamento:", detectar_outliers(colab["engajamento_medio"]))
print("Performance:", detectar_outliers(colab["performance_media"]))

Outliers após tratamento:
Horas extras: 363
Engajamento: 0
Performance: 0


As horas extras continuam sendo classificadas como outliers porque, embora ultrapassem os limites definidos pelo IQR, elas ainda estão dentro da faixa entre os percentis P1 e P99 usada na winsorização. Ou seja, são valores altos, mas não extremos segundo esse critério. Como esses picos refletem a dinâmica realista de sobrecarga no cenário simulado, decidi não aplicar winsorização nessa variável.

In [30]:
def detectar_outliers_iqr(serie):
    Q1, Q3 = serie.quantile(0.25), serie.quantile(0.75)
    IQR = Q3 - Q1
    li, ls = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    return int(((serie < li) | (serie > ls)).sum())

relatorio_qualidade = {
    "linhas_colaboradores": int(colab.shape[0]),
    "linhas_rh_mensal": int(rh_mensal.shape[0]),
    "linhas_promocoes": int(promocoes.shape[0]),
    "linhas_desligamentos": int(desligamentos.shape[0]),
    "outliers_horas_extras_iqr": detectar_outliers_iqr(colab["horas_extras_media"]),
    "outliers_engajamento_iqr": detectar_outliers_iqr(colab["engajamento_medio"]),
    "outliers_performance_iqr": detectar_outliers_iqr(colab["performance_media"]),
    "datas_deslig_invapos_limpeza": 0  # esperado após o tratamento
}

print("Relatório de Qualidade (resumo):")
for k,v in relatorio_qualidade.items():
    print(f"- {k}: {v}")

assert (colab["engajamento_medio"].between(0,100)).all(), "Engajamento fora de 0–100."
assert (colab["performance_media"].between(1,5)).all(), "Performance fora de 1–5."

Relatório de Qualidade (resumo):
- linhas_colaboradores: 2500
- linhas_rh_mensal: 60000
- linhas_promocoes: 350
- linhas_desligamentos: 121
- outliers_horas_extras_iqr: 363
- outliers_engajamento_iqr: 0
- outliers_performance_iqr: 0
- datas_deslig_invapos_limpeza: 0


In [31]:
# ==============================
# Duplicidades: detecção e tratamento
# ==============================

import pandas as pd

# ---- 1) dim_colaboradores: 'employee_id' deve ser único
dup_colab = colab.duplicated(subset=["employee_id"], keep=False)
qtd_dup_colab = dup_colab.sum()
print(f"[dim_colaboradores] Colaboradores com employee_id duplicado: {qtd_dup_colab}")

if qtd_dup_colab > 0:
    # Estratégia: manter o primeiro registro (ou o mais recente, se fizer sentido)
    # Aqui manteremos o primeiro para não introduzir regra não discutida
    colab = colab.drop_duplicates(subset=["employee_id"], keep="first").reset_index(drop=True)
    print(f"[dim_colaboradores] Duplicidades removidas. Linhas após limpeza: {colab.shape[0]}")

# ---- 2) fato_rh_mensal: chave composta (employee_id, mes_ano) deve ser única
rh_mensal["mes_ano"] = pd.to_datetime(rh_mensal["mes_ano"], errors="coerce")

dup_chave_mensal = rh_mensal.duplicated(subset=["employee_id", "mes_ano"], keep=False)
qtd_dup_mensal = dup_chave_mensal.sum()
print(f"[fato_rh_mensal] Registros duplicados por (employee_id, mes_ano): {qtd_dup_mensal}")

if qtd_dup_mensal > 0:
    # Estratégia de agregação para consolidar duplicados:
    # - métricas numéricas: média
    # - se houver campos categóricos, podemos manter o primeiro (aqui não há)
    numeric_cols = rh_mensal.select_dtypes(include=["number"]).columns.tolist()
    # garantir que não agregaremos a coluna employee_id em média por engano
    numeric_cols = [c for c in numeric_cols if c not in ["employee_id"]]

    rh_mensal = (
        rh_mensal.groupby(["employee_id", "mes_ano"], as_index=False)[numeric_cols].mean()
    )
    print(f"[fato_rh_mensal] Duplicidades consolidadas por média. Linhas após limpeza: {rh_mensal.shape[0]}")

# ---- 3) eventos_promocoes: pode haver múltiplas promoções por colaborador (em datas diferentes).
# Checamos apenas a duplicidade exata (employee_id, data, nivel_anterior, nivel_novo).
promocoes["data"] = pd.to_datetime(promocoes["data"], errors="coerce")

dup_promocoes_exatas = promocoes.duplicated(subset=["employee_id", "data", "nivel_anterior", "nivel_novo"], keep=False)
qtd_dup_promocoes = dup_promocoes_exatas.sum()
print(f"[eventos_promocoes] Duplicatas exatas de evento de promoção: {qtd_dup_promocoes}")

if qtd_dup_promocoes > 0:
    promocoes = promocoes.drop_duplicates(subset=["employee_id", "data", "nivel_anterior", "nivel_novo"], keep="first").reset_index(drop=True)
    print(f"[eventos_promocoes] Duplicidades exatas removidas. Linhas após limpeza: {promocoes.shape[0]}")

# ---- 4) eventos_desligamentos: no máximo 1 desligamento por colaborador
desligamentos["data_desligamento"] = pd.to_datetime(desligamentos["data_desligamento"], errors="coerce")

qtd_desligamentos_por_colab = desligamentos.groupby("employee_id").size()
colab_com_mais_de_um_desligamento = qtd_desligamentos_por_colab[qtd_desligamentos_por_colab > 1]

print(f"[eventos_desligamentos] Colaboradores com >1 desligamento: {colab_com_mais_de_um_desligamento.shape[0]}")

if colab_com_mais_de_um_desligamento.shape[0] > 0:
    # Estratégia: manter o desligamento mais recente (maior data)
    desligamentos = (
        desligamentos.sort_values(["employee_id", "data_desligamento"])
        .groupby("employee_id", as_index=False)
        .tail(1)
        .reset_index(drop=True)
    )
    print(f"[eventos_desligamentos] Consolidado para 1 desligamento por colaborador. Linhas após limpeza: {desligamentos.shape[0]}")

# ---- 5) Checagem de integridade referencial (FKs): fatos/eventos devem existir em dim
ids_dim = set(colab["employee_id"])

ids_rh = set(rh_mensal["employee_id"])
ids_prom = set(promocoes["employee_id"])
ids_desl = set(desligamentos["employee_id"])

faltantes_rh = ids_rh - ids_dim
faltantes_prom = ids_prom - ids_dim
faltantes_desl = ids_desl - ids_dim

print(f"[FK] employee_id em fato_rh_mensal inexistentes na dimensão: {len(faltantes_rh)}")
print(f"[FK] employee_id em eventos_promocoes inexistentes na dimensão: {len(faltantes_prom)}")
print(f"[FK] employee_id em eventos_desligamentos inexistentes na dimensão: {len(faltantes_desl)}")

# Estratégia: remover linhas órfãs (se houver)
if len(faltantes_rh) > 0:
    rh_mensal = rh_mensal[~rh_mensal["employee_id"].isin(faltantes_rh)].reset_index(drop=True)
    print("[FK] Removidos registros órfãos de fato_rh_mensal.")

if len(faltantes_prom) > 0:
    promocoes = promocoes[~promocoes["employee_id"].isin(faltantes_prom)].reset_index(drop=True)
    print("[FK] Removidos registros órfãos de eventos_promocoes.")

if len(faltantes_desl) > 0:
    desligamentos = desligamentos[~desligamentos["employee_id"].isin(faltantes_desl)].reset_index(drop=True)
    print("[FK] Removidos registros órfãos de eventos_desligamentos.")

[dim_colaboradores] Colaboradores com employee_id duplicado: 0
[fato_rh_mensal] Registros duplicados por (employee_id, mes_ano): 0
[eventos_promocoes] Duplicatas exatas de evento de promoção: 0
[eventos_desligamentos] Colaboradores com >1 desligamento: 0
[FK] employee_id em fato_rh_mensal inexistentes na dimensão: 0
[FK] employee_id em eventos_promocoes inexistentes na dimensão: 0
[FK] employee_id em eventos_desligamentos inexistentes na dimensão: 0


In [32]:
# Asserts críticos (levantam erro se falhar)
assert colab["employee_id"].is_unique, "employee_id duplicado em dim_colaboradores."
assert not rh_mensal.duplicated(subset=["employee_id", "mes_ano"]).any(), "(employee_id, mes_ano) duplicado em fato_rh_mensal."
assert (desligamentos.groupby("employee_id").size().max() <= 1), "Mais de um desligamento por colaborador."
assert (colab["engajamento_medio"].between(0,100)).all(), "Engajamento fora de 0–100."
assert (colab["performance_media"].between(1,5)).all(), "Performance fora de 1–5."

# Prints de sucesso (tornam o notebook 'falante' sem poluir)
print("✔ unicidade em dim_colaboradores OK")
print("✔ chave (employee_id, mes_ano) única em fato_rh_mensal OK")
print("✔ no máximo um desligamento por colaborador OK")
print("✔ limites naturais: engajamento (0–100) e performance (1–5) OK")

✔ unicidade em dim_colaboradores OK
✔ chave (employee_id, mes_ano) única em fato_rh_mensal OK
✔ no máximo um desligamento por colaborador OK
✔ limites naturais: engajamento (0–100) e performance (1–5) OK
